In [1]:
import sys
import pandas as pd
sys.path.append('../src')
from battle_engine import BattlePokemon, BattleState, TurnEngine
from synergy_scorer import TeamSynergyScorer

In [2]:
df_pokemon = pd.read_csv('../data/processed/clustered_pokemon_data.csv').set_index('name', drop=False)
df_moves = pd.read_csv('../data/raw/showdown_moves.csv').set_index('name', drop=False)
type_chart = TeamSynergyScorer().type_chart

# 2. Instantiate 4 Pokémon
incineroar = BattlePokemon('incineroar', df_pokemon.loc['incineroar'], [], 'Intimidate')
rillaboom = BattlePokemon('rillaboom', df_pokemon.loc['rillaboom'], [], 'Grassy Surge')
flutter_mane = BattlePokemon('flutter-mane', df_pokemon.loc['flutter-mane'], [], 'Protosynthesis')
chi_yu = BattlePokemon('chi-yu', df_pokemon.loc['chi-yu'], [], 'Beads of Ruin')

In [5]:
state = BattleState()
state.team_a_active = [incineroar, rillaboom]
state.team_b_active = [flutter_mane, chi_yu]

engine = TurnEngine(state, type_chart)

In [6]:
fake_out = df_moves.loc['Fake Out'].to_dict()
dazzling_gleam = df_moves.loc['Dazzling Gleam'].to_dict()
heat_wave = df_moves.loc['Heat Wave'].to_dict()

In [7]:
actions = [
    # Incineroar uses Fake Out on Chi-Yu (Priority +3)
    {'user': incineroar, 'targets': [chi_yu], 'move': fake_out},
    # Flutter Mane uses Spread Move hitting both opponents
    {'user': flutter_mane, 'targets': [incineroar, rillaboom], 'move': dazzling_gleam},
    # Chi-Yu uses Spread Move
    {'user': chi_yu, 'targets': [incineroar, rillaboom], 'move': heat_wave},
    # Rillaboom attacks Flutter Mane
    {'user': rillaboom, 'targets': [flutter_mane], 'move': df_moves.loc['Wood Hammer'].to_dict()}
]

In [8]:
print("--- EXECUTING VGC DOUBLES TURN ---")
logs = engine.execute_turn(actions)

for log in logs:
    print(log)

--- EXECUTING VGC DOUBLES TURN ---
incineroar used Fake Out! Dealt 22 to chi-yu. (140/162 HP remaining)
flutter-mane used Dazzling Gleam! Dealt 47 to incineroar. (155/202 HP remaining)
flutter-mane used Dazzling Gleam! Dealt 58 to rillaboom. (149/207 HP remaining)
chi-yu used Heat Wave! Dealt 27 to incineroar. (128/202 HP remaining)
chi-yu used Heat Wave! Dealt 141 to rillaboom. (8/207 HP remaining)
rillaboom used Wood Hammer! Dealt 114 to flutter-mane. (48/162 HP remaining)
